# AMPR — Kaggle Run

**Environment:** Kaggle — Python 3.12.12, 2×T4 GPU (15 GB VRAM mỗi card)

## Pipeline

```
Phase 0 ── Kiểm tra GPU (2×T4)
Phase 1 ── Clone repo & cài thư viện
Phase 2 ── Symlink Kaggle dataset → data/
Phase 3 ── ProstT5 structural embeddings (encoder-only, fp16, batch=8) ~1.5h
Phase 4 ── Verify embeddings + Update configs
Phase 5 ── Train: Molecular Function (MF)  ~1.5h
Phase 6 ── Train: Biological Process (BP)  ~3-4h
Phase 7 ── Train: Cellular Component (CC)  ~1h
Phase 8 ── Kết quả & tổng hợp
```

## Chuẩn bị trước khi chạy

Upload các file sau lên **Kaggle Dataset** (tạo dataset "ampr-data"):

| File | Tạo ở | Size |
|---|---|---|
| `embeddings/seq_embeddings.npy` | Colab Phase 5.2 | ~170 MB |
| `embeddings/ppi_embeddings.npy` | Colab Phase 5.4 | ~45 MB |
| `embeddings/go_emb_mf/bp/cc.npy` | Colab Phase 5.5 | ~8 MB |
| `labels_mf/bp/cc.npy` | Colab Phase 6.2 | ~100–300 MB |
| `dag_matrices/dag_matrix_mf/bp/cc.npy` | Colab Phase 6.1 | nhỏ |
| `splits.json` | Colab Phase 6.3 | nhỏ |
| `protein_order.json` | Colab Phase 5.1 | nhỏ |
| `raw/nrPDB-GO_sequences.fasta` | Colab Phase 4 | ~14 MB |

> `struct_embeddings.npy` **không** cần upload — sẽ tạo ngay trên Kaggle (Phase 3).

---
## Phase 0 — Kiểm tra GPU

In [ ]:
import subprocess, os

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('[GPU CHECK]')
print(result.stdout)

import torch
n_gpu = torch.cuda.device_count()
print(f'torch.cuda.device_count(): {n_gpu}')
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

if n_gpu < 2:
    print('[WARN] Chỉ có 1 GPU — ProstT5 vẫn chạy được nhưng tận dụng 1 GPU thôi.')
else:
    print('[OK] 2×T4 xác nhận — ProstT5 encoder-only fp16 chạy trên GPU 0 (batch=8)')

---
## Phase 1 — Clone repo & cài thư viện

In [ ]:
REPO_URL = 'https://github.com/hungithust/datn_protein_function.git'
WORK_DIR = '/kaggle/working/ampr'

if not os.path.exists(WORK_DIR):
    !git clone -q {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull -q

os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Kaggle có torch pre-installed — chỉ cài thêm packages còn thiếu
!pip install -q transformers==4.41.2 biopython==1.84 obonet==1.0.0 pyyaml==6.0.1 tqdm==4.66.4
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html

import sys, torch, transformers, numpy
print(f'Python       : {sys.version.split()[0]}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {numpy.__version__}')
print('✓ Thư viện sẵn sàng')

---
## Phase 2 — Symlink Kaggle dataset → data/

Thay `YOUR_DATASET_NAME` bằng tên dataset đã tạo trên Kaggle.

In [ ]:
KAGGLE_DATASET = '/kaggle/input/YOUR_DATASET_NAME'  # <-- thay tên dataset

print('[DATASET FILES]')
!ls -lh {KAGGLE_DATASET}/

DATA_DIR = f'{WORK_DIR}/data'
if not os.path.exists(DATA_DIR):
    os.symlink(KAGGLE_DATASET, DATA_DIR)
    print(f'Symlink: {DATA_DIR} → {KAGGLE_DATASET}')
else:
    print(f'data/ đã tồn tại: {DATA_DIR}')

# Verify files
required = [
    'data/embeddings/seq_embeddings.npy',
    'data/embeddings/ppi_embeddings.npy',
    'data/embeddings/go_emb_mf.npy',
    'data/embeddings/go_emb_bp.npy',
    'data/embeddings/go_emb_cc.npy',
    'data/labels_mf.npy',
    'data/labels_bp.npy',
    'data/labels_cc.npy',
    'data/dag_matrices/dag_matrix_mf.npy',
    'data/dag_matrices/dag_matrix_bp.npy',
    'data/dag_matrices/dag_matrix_cc.npy',
    'data/splits.json',
    'data/protein_order.json',
    'data/raw/nrPDB-GO_sequences.fasta',
]
print('\n[VERIFY]')
all_ok = True
for f in required:
    ok = os.path.exists(f)
    print(f'  {"✓" if ok else "✗ MISSING"}  {f}')
    if not ok:
        all_ok = False

print('\n[OK] Tất cả files có mặt' if all_ok else '\n[WARN] Có file bị thiếu — kiểm tra lại Kaggle Dataset')

---
## Phase 3 — ProstT5 Structural Embeddings (Kaggle 2×T4)

Dùng **encoder-only** + **fp16** + **batch=8** thay vì full T5 forward với batch=2 trên Colab:
- Colab T4: batch=2 → **8–10h**
- Kaggle T4 fp16 encoder-only: batch=8 → **~1.5h**

Output: `data/embeddings/struct_embeddings.npy` shape=(44422, 1024)

In [ ]:
import json, numpy as np, torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from Bio import SeqIO

STRUCT_EMB_PATH = 'data/embeddings/struct_embeddings.npy'

if os.path.exists(STRUCT_EMB_PATH):
    emb = np.load(STRUCT_EMB_PATH)
    print(f'[SKIP] {STRUCT_EMB_PATH} đã tồn tại. shape={emb.shape}')
else:
    # Load FASTA + protein_order
    FASTA_PATH    = 'data/raw/nrPDB-GO_sequences.fasta'
    protein_order = json.load(open('data/protein_order.json'))
    prot2idx      = {p: i for i, p in enumerate(protein_order)}

    records   = list(SeqIO.parse(FASTA_PATH, 'fasta'))
    sequences = [str(r.seq) for r in records]
    seq_ids   = [r.id     for r in records]
    print(f'[FASTA] Loaded {len(sequences)} sequences')

    # ProstT5: encoder-only, fp16
    print('[LOAD] Loading ProstT5 (fp16, encoder-only)...')
    device    = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('Rostlab/ProstT5', do_lower_case=False, use_fast=False)
    model     = AutoModel.from_pretrained('Rostlab/ProstT5').half().to(device)
    model.eval()
    print(f'[LOAD] Model on {device} (fp16)')

    BATCH_SIZE = 8   # fp16 + 1×T4 15GB: safe với max_length=512
    all_embeddings = []

    print(f'[RUN] Encoding {len(sequences)} sequences (batch={BATCH_SIZE})...')
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc='ProstT5'):
            batch_seqs   = sequences[i:i+BATCH_SIZE]
            batch_spaced = ['<AA2fold> ' + ' '.join(s) for s in batch_seqs]

            encoded = tokenizer(
                batch_spaced,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)

            # Encoder-only: ~40% ít VRAM hơn full T5 forward
            enc_out = model.encoder(
                input_ids=encoded['input_ids'],
                attention_mask=encoded['attention_mask'],
            )
            hidden = enc_out.last_hidden_state.float()   # cast fp16→fp32
            mask   = encoded['attention_mask'].unsqueeze(-1).float()
            emb    = (hidden * mask).sum(dim=1) / mask.sum(dim=1)   # (B, 1024)
            all_embeddings.append(emb.cpu().numpy())

    raw_emb = np.concatenate(all_embeddings, axis=0).astype(np.float32)

    # Reorder theo protein_order (FASTA order có thể khác)
    N = len(protein_order)
    struct_emb = np.zeros((N, 1024), dtype=np.float32)
    for fasta_idx, pid in enumerate(seq_ids):
        if pid in prot2idx:
            struct_emb[prot2idx[pid]] = raw_emb[fasta_idx]

    print(f'[OUT]  struct_embeddings shape: {struct_emb.shape}')
    np.save(STRUCT_EMB_PATH, struct_emb)
    print(f'[SAVED] {STRUCT_EMB_PATH} ({struct_emb.nbytes/1e6:.1f} MB)')

    del model, all_embeddings, raw_emb
    torch.cuda.empty_cache()

---
## Phase 4 — Verify embeddings + Update configs

In [ ]:
import json, numpy as np

protein_order = json.load(open('data/protein_order.json'))
N      = len(protein_order)
splits = json.load(open('data/splits.json'))

print(f'[INFO] N proteins = {N:,}')
print(f'[INFO] Splits: train={len(splits["train"]):,}, valid={len(splits["valid"]):,}, test={len(splits["test"]):,}')

checks = [
    ('seq_embeddings.npy',    (N, 1024)),
    ('struct_embeddings.npy', (N, 1024)),
    ('ppi_embeddings.npy',    (N, None)),
    ('go_emb_mf.npy',         (None, 768)),
    ('go_emb_bp.npy',         (None, 768)),
    ('go_emb_cc.npy',         (None, 768)),
]

print('\n[VERIFY EMBEDDINGS]')
all_ok  = True
ppi_dim = None
for fname, expected in checks:
    path = f'data/embeddings/{fname}'
    if not os.path.exists(path):
        print(f'  ✗ MISSING: {path}')
        all_ok = False
        continue
    arr = np.load(path)
    ok  = all(exp is None or arr.shape[dim] == exp for dim, exp in enumerate(expected))
    print(f'  {"✓" if ok else "✗ SHAPE MISMATCH"}  {fname}: {arr.shape}')
    if not ok:
        all_ok = False
    if 'ppi_embeddings' in fname:
        ppi_dim = int(arr.shape[1])

print('\n[OK] Tất cả embeddings hợp lệ' if all_ok else '\n[ERROR] Kiểm tra lại các file bị thiếu/sai shape')
print(f'[INFO] ppi_dim detected = {ppi_dim}')

In [ ]:
import yaml

for branch in ['mf', 'bp', 'cc']:
    os.makedirs(f'{WORK_DIR}/checkpoints/{branch}', exist_ok=True)
os.makedirs(f'{WORK_DIR}/logs',    exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

def update_config_paths(config_path, work_dir, ppi_dim=None):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    for key in cfg['data']:
        if isinstance(cfg['data'][key], str) and cfg['data'][key].startswith('data/'):
            cfg['data'][key] = os.path.join(work_dir, cfg['data'][key])

    for key in cfg['output']:
        if isinstance(cfg['output'][key], str):
            cfg['output'][key] = os.path.join(work_dir, cfg['output'][key])
            os.makedirs(os.path.dirname(cfg['output'][key]), exist_ok=True)

    if ppi_dim is not None:
        cfg['model']['ppi_dim'] = int(ppi_dim)

    out_path = config_path.replace('.yaml', '_kaggle.yaml')
    with open(out_path, 'w') as f:
        yaml.dump(cfg, f)
    return out_path

mf_config = update_config_paths(f'{WORK_DIR}/configs/mf.yaml', WORK_DIR, ppi_dim)
bp_config = update_config_paths(f'{WORK_DIR}/configs/bp.yaml', WORK_DIR, ppi_dim)
cc_config = update_config_paths(f'{WORK_DIR}/configs/cc.yaml', WORK_DIR, ppi_dim)

print('Kaggle configs generated:')
for c in [mf_config, bp_config, cc_config]:
    print(f'  {c}')

---
## Phase 5 — Train: Molecular Function (MF)

~611 GO terms. Ước tính: **~1.5h** trên Kaggle.

In [ ]:
print('=' * 60)
print('TRAINING: Molecular Function (MF)')
print('=' * 60)
!python {WORK_DIR}/main.py --config {mf_config} --seed 42
print('\n[✓] MF done!')
!ls -lh {WORK_DIR}/checkpoints/mf/

---
## Phase 6 — Train: Biological Process (BP)

~1810 GO terms — training lâu nhất. Ước tính: **~3–4h** trên Kaggle.

In [ ]:
print('=' * 60)
print('TRAINING: Biological Process (BP)')
print('=' * 60)
!python {WORK_DIR}/main.py --config {bp_config} --seed 42
print('\n[✓] BP done!')
!ls -lh {WORK_DIR}/checkpoints/bp/

---
## Phase 7 — Train: Cellular Component (CC)

~371 GO terms — nhanh nhất. Ước tính: **~1h** trên Kaggle.

In [ ]:
print('=' * 60)
print('TRAINING: Cellular Component (CC)')
print('=' * 60)
!python {WORK_DIR}/main.py --config {cc_config} --seed 42
print('\n[✓] CC done!')
!ls -lh {WORK_DIR}/checkpoints/cc/

---
## Phase 8 — Kết quả & Biểu đồ

Trainer tự động lưu sau mỗi branch:
- `results/test_metrics_{branch}.json` — Fmax, AUPRC, Smin, AUROC, Coverage
- `results/training_history_{branch}.json` — loss + val Fmax từng epoch
- `results/plots/` — tất cả biểu đồ (PNG)
- `results/test_predictions_{branch}.npz` — y_true / y_pred thô để phân tích offline
- `checkpoints/{branch}/best.pt` — model tốt nhất

In [ ]:
import json, os
import numpy as np

BRANCHES = ['mf', 'bp', 'cc']
RESULTS  = f'{WORK_DIR}/results'

# ── 1. Comparison table ──────────────────────────────────────────────────────
COLS  = ['fmax', 'auprc', 'smin', 'micro_auroc', 'macro_auroc', 'coverage']
HEADS = ['Fmax', 'AUPRC', 'Smin↓', 'AUROC micro', 'AUROC macro', 'Coverage']
WIDTH = 11

hdr = f"{'Branch':>8} │ " + " │ ".join(f"{h:^{WIDTH}}" for h in HEADS)
sep = "─" * len(hdr)
print(sep); print(hdr); print(sep)

for branch in BRANCHES:
    path = f'{RESULTS}/test_metrics_{branch}.json'
    if not os.path.exists(path):
        print(f'{branch.upper():>8} │  (not found — training may not have completed)')
        continue
    m = json.load(open(path))
    vals = " │ ".join(f"{m.get(c, 0.0):^{WIDTH}.4f}" for c in COLS)
    print(f'{branch.upper():>8} │ {vals}')

print(sep)

In [ ]:
from IPython.display import Image, display, HTML
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PLOT_TYPES = [
    ('training_curves',  'Training Loss + Val Fmax'),
    ('alpha_evolution',  'Modality Gating Weights (α)'),
    ('pr_curve',         'Precision-Recall Curve'),
    ('roc_curve',        'ROC Curve'),
    ('threshold_sweep',  'Threshold Sweep'),
]

PLOTS_DIR = f'{RESULTS}/plots'

for branch in BRANCHES:
    available = [(pt, title) for pt, title in PLOT_TYPES
                 if os.path.exists(f'{PLOTS_DIR}/{pt}_{branch}.png')]
    if not available:
        print(f'[{branch.upper()}] No plots found — training may not have completed.')
        continue

    display(HTML(f'<h3 style="margin-top:24px">{branch.upper()} branch</h3>'))
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, (pt, title) in zip(axes, available):
        img = mpimg.imread(f'{PLOTS_DIR}/{pt}_{branch}.png')
        ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Training history table (best epoch per branch) ───────────────────────────
print('\n[TRAINING HISTORY — Best Val Epoch per Branch]')
print(f"{'Branch':>8} │ {'Best Epoch':>10} │ {'Val Fmax':>9} │ {'Train Loss':>10} │ {'BCE':>8} │ {'DAG':>8}")
print('─' * 70)
for branch in BRANCHES:
    path = f'{RESULTS}/training_history_{branch}.json'
    if not os.path.exists(path):
        print(f'{branch.upper():>8} │  (not found)')
        continue
    history = json.load(open(path))
    best = max(history, key=lambda h: h['val_fmax'])
    print(f"{branch.upper():>8} │ {best['epoch']:>10} │ {best['val_fmax']:>9.4f} │ "
          f"{best['train_loss']:>10.4f} │ {best['bce']:>8.4f} │ {best['dag']:>8.4f}")
print('─' * 70)

print('\n[CHECKPOINTS]')
for branch in BRANCHES:
    ckpt = f'{WORK_DIR}/checkpoints/{branch}/best.pt'
    if os.path.exists(ckpt):
        import torch
        data = torch.load(ckpt, map_location='cpu')
        size = os.path.getsize(ckpt) / 1e6
        print(f'  {branch.upper()}: epoch={data["epoch"]}, val Fmax={data["fmax"]:.4f}, size={size:.1f} MB')
    else:
        print(f'  {branch.upper()}: not found')

print('\n[OUTPUT FILES]')
print(f'  Plots  : {RESULTS}/plots/')
print(f'  Metrics: {RESULTS}/test_metrics_{{mf,bp,cc}}.json')
print(f'  History: {RESULTS}/training_history_{{mf,bp,cc}}.json')
print(f'  Preds  : {RESULTS}/test_predictions_{{mf,bp,cc}}.npz')
print(f'  Models : {WORK_DIR}/checkpoints/{{mf,bp,cc}}/best.pt')
print('\nDownload from the Output tab of this Kaggle session.')

---
## So sánh Kaggle vs Colab

| Điểm | Colab | Kaggle |
|---|---|---|
| GPU | 1×T4 15 GB | 2×T4 15 GB |
| Session | ~4–6h (free) | 12h |
| ProstT5 batch | 2 (~8–10h) | 8 fp16 (~1.5h) |
| ProstT5 method | full T5 forward | encoder-only |
| PPI embeddings | DeepGO pkl (Phase 5.4) | từ Kaggle dataset |
| Storage | Google Drive | Kaggle Output tab |

**ppi_dim note:** Config tự động sync `ppi_dim` từ shape thực tế của `ppi_embeddings.npy`  
→ 256 (DeepGO) hoặc 128 (zero fallback).